In [1]:
import torch

a=torch.tensor([[1.,2.],[3.,4.]])
b= torch.ones(2,2)
print(a + b)
print(a @ b)          # matrix multiply
print(a.mean(), a.shape)
print(a.reshape(4), a.reshape(4).shape)

tensor([[2., 3.],
        [4., 5.]])
tensor([[3., 3.],
        [7., 7.]])
tensor(2.5000) torch.Size([2, 2])
tensor([1., 2., 3., 4.]) torch.Size([4])


In [2]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1
y.backward()
print(x.grad)         # → 8.0

tensor(8.)


In [3]:
# secret truth: y = 3x + 7. Can the machine discover it?
X = torch.randn(100, 1)
y_true = 3 * X + 7 + 0.1 * torch.randn(100, 1)

w = torch.zeros(1, requires_grad=True)     # the two knobs, starting dumb
b = torch.zeros(1, requires_grad=True)

for step in range(200):
    y_pred = w * X + b                     # 1. GUESS with current knobs
    loss = ((y_pred - y_true)**2).mean()   # 2. how WRONG? (mean squared error)
    loss.backward()                        # 3. which way should each knob turn?
    with torch.no_grad():                  # 4. turn them (no tracking while adjusting)
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_(); b.grad.zero_()         # 5. wipe the slate for next round
    if step % 50 == 0:
        print(f"step {step}: loss {loss.item():.4f}, w {w.item():.2f}, b {b.item():.2f}")

step 0: loss 52.6205, w 0.40, b 1.33
step 50: loss 0.0118, w 2.99, b 6.99
step 100: loss 0.0118, w 2.99, b 6.99
step 150: loss 0.0118, w 2.99, b 6.99


In [4]:
import torch.nn as nn

model = nn.Linear(1, 1)                                # w and b, packaged
opt = torch.optim.SGD(model.parameters(), lr=0.1)      # the knob-turner
loss_fn = nn.MSELoss()                                 # the wrongness-measurer

for step in range(200):
    pred = model(X)               # 1. forward (the guess)
    loss = loss_fn(pred, y_true)  # 2. loss
    opt.zero_grad()               # 3. wipe slate
    loss.backward()               # 4. gradients
    opt.step()                    # 5. turn knobs
print(model.weight.item(), model.bias.item())   # ≈ 3 and 7 again

2.989853858947754 6.994680404663086


In [5]:
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd

feats = pd.read_parquet("../data/processed/features/TSLA.parquet")
FEATURES = ["ret_1d","ret_2d","ret_3d","ret_5d","ret_10d",
            "price_vs_ma20","ma20_vs_ma50","vol_20d","rsi_14","macd","volume_z"]
tr = feats[feats.index < "2024-01-01"]

mu, sd = tr[FEATURES].mean(), tr[FEATURES].std()       # scaling stats: TRAIN ONLY (the promise!)
Xt = torch.tensor(((tr[FEATURES] - mu) / sd).values, dtype=torch.float32)
yt = torch.tensor(tr["target_1d"].astype(int).values, dtype=torch.float32).unsqueeze(1)

loader = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print(xb.shape, yb.shape)      # → [64, 11] and [64, 1]

torch.Size([64, 11]) torch.Size([64, 1])


In [6]:
model = nn.Sequential(
    nn.Linear(11, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(30):
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 10 == 0:
        print(f"epoch {epoch}: loss {loss.item():.4f}")

# exam time — 2025 rows, scaled with TRAIN's mu/sd:
te = feats[feats.index >= "2025-01-01"]
Xe = torch.tensor(((te[FEATURES] - mu) / sd).values, dtype=torch.float32)
with torch.no_grad():
    acc = ((torch.sigmoid(model(Xe)) > 0.5).float().squeeze().numpy()
           == te["target_1d"].astype(int).values).mean()
print(f"MLP test accuracy: {acc:.3f}  (compare: baseline & this morning's XGBoost)")


epoch 0: loss 0.6842
epoch 10: loss 0.6881
epoch 20: loss 0.6747
MLP test accuracy: 0.520  (compare: baseline & this morning's XGBoost)


In [ ]:
# 🎓 GRADUATION — the five-step training loop, FROM MEMORY
# Rules: no scrolling up. Fill the 5 lines below the comments. ~60 seconds.
# (You have typed this loop twice already today — it IS in your fingers.)

for epoch in range(30):
    for xb, yb in loader:
        # 1. FORWARD — make the guess with current knobs
        pred=model(xb)
        # 2. LOSS — measure how wrong the guess is
        loss=loss_fn(pred,yb)
        # 3. ZERO_GRAD — wipe the slate (gradients accumulate otherwise!)
        opt.zero_grad()
        # 4. BACKWARD compute which way each knob should turn
        loss.backward()
        # 5. STEP — turn the knobs
        opt.step()

